# Travel Reimbursement Approval Agent

This notebook implements an AI-assisted Travel Reimbursement Approval Agent.

The agent evaluates employee travel reimbursement claims against:
- Travel reimbursement policy
- Receipt requirements
- Category eligibility
- Per-diem/category limits
- Approval thresholds
- Policy exceptions

It produces one of four decisions:

- APPROVE
- PARTIAL_APPROVE
- REJECT
- MANUAL_REVIEW

The solution combines deterministic Python tools with an LLM-based reasoning layer to provide reliable, auditable decisions.

In [25]:
import os
import json
from typing import Any, Dict, List, Literal, Optional
from dataclasses import dataclass
from datetime import date, datetime

import pandas as pd
import matplotlib.pyplot as plt

from pydantic import BaseModel, Field

In [26]:
import os
from getpass import getpass

MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "openai")
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured.")

print(f"Model provider: {MODEL_PROVIDER}")
print(f"Model: {MODEL_NAME}")

OpenAI API key configured.
Model provider: openai
Model: gpt-4o-mini


In [27]:
# ============================================
# Data Models
# ============================================

Decision = Literal[
    "APPROVE",
    "PARTIAL_APPROVE",
    "REJECT",
    "MANUAL_REVIEW",
]


@dataclass
class ClaimItem:
    category: str
    description: str
    amount: float
    receipt_attached: bool
    quantity: Optional[float] = None


@dataclass
class Claim:
    claim_id: str
    employee: str
    trip_start: str
    trip_end: str
    submitted: str
    business_purpose: str
    items: List[ClaimItem]


def claim_total(claim: Claim) -> float:
    return round(
        sum(item.amount for item in claim.items),
        2
    )


def parse_date(value: str) -> date:
    return datetime.strptime(
        value,
        "%Y-%m-%d"
    ).date()

In [7]:
# ============================================
# Policy Knowledge Base
# ============================================

POLICIES = {

    "POL-CAT-01": {
        "title": "Eligible categories",
        "text": (
            "Airfare (economy class only), lodging, meals, "
            "ground transport, and conference/registration fees "
            "are reimbursable when incurred for a documented "
            "business purpose."
        ),
    },

    "POL-CAT-02": {
        "title": "Ineligible items",
        "text": (
            "Alcohol and minibar charges, spa, gym and personal "
            "entertainment, in-room movies, personal shopping, "
            "gifts, traffic fines, penalties, late fees and "
            "personal non-business expenses are never reimbursable."
        ),
    },

    "POL-PD-01": {
        "title": "Meals limit",
        "text": (
            "Meals have a maximum reimbursement of $75 per day."
        ),
    },

    "POL-PD-02": {
        "title": "Lodging limit",
        "text": (
            "Lodging has a maximum reimbursement of $200 per night."
        ),
    },

    "POL-PD-03": {
        "title": "Ground transport limit",
        "text": (
            "Ground transport has a maximum reimbursement of $50 per day."
        ),
    },

    "POL-AIR-01": {
        "title": "Airfare class",
        "text": (
            "Only economy class airfare is reimbursable. "
            "Business/first-class fares are policy exceptions "
            "and must be routed to Manual Review."
        ),
    },

    "POL-RCT-01": {
        "title": "Receipt requirement",
        "text": (
            "Any single line item greater than $25 requires an "
            "attached itemized receipt. Airfare and lodging "
            "always require a receipt."
        ),
    },

    "POL-RCT-02": {
        "title": "Missing receipt handling",
        "text": (
            "If a required receipt is missing, the claim must "
            "be routed to Manual Review."
        ),
    },

    "POL-APR-01": {
        "title": "Auto-approve tier",
        "text": (
            "Total reimbursable amount <= $500 may be auto-approved "
            "if fully compliant."
        ),
    },

    "POL-APR-02": {
        "title": "Manager tier",
        "text": (
            "Total > $500 and <= $2,000 is approvable when fully compliant."
        ),
    },

    "POL-APR-03": {
        "title": "Director / Manual Review tier",
        "text": (
            "Total > $2,000 exceeds agent approval authority "
            "and must be routed to Manual Review."
        ),
    },

    "POL-TIME-01": {
        "title": "Submission window",
        "text": (
            "Claims must be submitted within 30 days of the "
            "expense date. Late claims are routed to Manual Review."
        ),
    },
}

In [8]:
print(f"Loaded {len(POLICIES)} policy rules.")

Loaded 12 policy rules.


In [28]:
## Sample claims data for testing

CLAIMS = [
    {
        "claim_id": "CLM-001",""
        "employee": "A. Rivera",
        "description": "Attend 2-day industry conference (business)",
        "trip_start": "2026-06-10",
        "trip_end": "2026-06-12",
        "submitted_date": "2026-06-20",
        "items": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": 420.00,
                "receipt_attached": True
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $180",
                "amount": 360.00,
                "receipt_attached": True,
                "nights": 2
            },
            {
                "category": "meals",
                "description": "Meals, 3 days @ ~$60/day",
                "amount": 180.00,
                "receipt_attached": True,
                "days": 3
            },
            {
                "category": "conference_fees",
                "description": "Conference registration",
                "amount": 150.00,
                "receipt_attached": True
            }
        ]
    },

    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "description": "Weekend hotel stay",
        "trip_start": "2026-06-14",
        "trip_end": "2026-06-15",
        "submitted_date": "2026-06-25",
        "items": [
            {
                "category": "spa",
                "description": "Hotel spa package",
                "amount": 300.00,
                "receipt_attached": True
            },
            {
                "category": "minibar",
                "description": "In-room minibar",
                "amount": 80.00,
                "receipt_attached": True
            }
        ]
    },

    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "description": "Client site visit (business)",
        "trip_start": "2026-06-08",
        "trip_end": "2026-06-10",
        "submitted_date": "2026-06-22",
        "items": [
            {
                "category": "airfare",
                "description": "Round-trip economy airfare",
                "amount": 300.00,
                "receipt_attached": True
            },
            {
                "category": "lodging",
                "description": "Hotel, 2 nights @ $250",
                "amount": 500.00,
                "receipt_attached": True,
                "nights": 2
            },
            {
                "category": "meals",
                "description": "Meals, 2 days @ $70/day",
                "amount": 140.00,
                "receipt_attached": True,
                "days": 2
            }
        ]
    },

    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "description": "International vendor negotiation (business)",
        "trip_start": "2026-06-16",
        "trip_end": "2026-06-18",
        "submitted_date": "2026-06-28",
        "items": [
            {
                "category": "airfare",
                "description": "Business-class international airfare",
                "amount": 2400.00,
                "receipt_attached": True
            },
            {
                "category": "lodging",
                "description": "Hotel, 3 nights",
                "amount": 600.00,
                "receipt_attached": False,
                "nights": 3
            }
        ]
    },

    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "description": "Client dinner / business development",
        "trip_start": "2026-06-11",
        "trip_end": "2026-06-11",
        "submitted_date": "2026-06-24",
        "items": [
            {
                "category": "meals",
                "description": "Client dinner for 4 (business development)",
                "amount": 220.00,
                "receipt_attached": False,
                "days": 1
            }
        ]
    }
]

print(f"Loaded {len(CLAIMS)} claims.")

Loaded 5 claims.


In [10]:
class ClaimItem(BaseModel):
    category: str
    description: str
    amount: float = Field(ge=0)
    receipt_attached: bool
    nights: Optional[int] = None
    days: Optional[int] = None


class Claim(BaseModel):
    claim_id: str
    employee: str
    description: str
    trip_start: str
    trip_end: str
    submitted_date: str
    items: list[ClaimItem]


validated_claims = [Claim(**claim) for claim in CLAIMS]

print("All claims passed structural validation.")

All claims passed structural validation.


In [29]:
def policy_lookup(claim):
    """
    Retrieve relevant policies from the policies knowledge base
    created earlier in the notebook.
    """

    categories = {
        item["category"].lower()
        for item in claim.get("items", [])
    }

    relevant_policy_ids = {
        "POL-CAT-01",
        "POL-CAT-02",
        "POL-RCT-01",
        "POL-RCT-02",
        "POL-APR-01",
        "POL-APR-02",
        "POL-APR-03",
        "POL-TIME-01",
    }

    if "meals" in categories:
        relevant_policy_ids.add("POL-PD-01")

    if "lodging" in categories:
        relevant_policy_ids.add("POL-PD-02")

    if "ground_transport" in categories:
        relevant_policy_ids.add("POL-PD-03")

    if "airfare" in categories:
        relevant_policy_ids.add("POL-AIR-01")

    relevant_rules = {
        policy_id: POLICIES[policy_id]
        for policy_id in relevant_policy_ids
        if policy_id in POLICIES
    }

    return {
        "claim_id": claim["claim_id"],
        "rules": relevant_rules,
        "policy_refs": sorted(relevant_rules.keys())
    }

In [30]:
#  Receipt Completeness Tool

def receipt_completeness_check(claim):
    """
    Check whether all required receipts are attached.

    Policy:
    - Airfare and lodging always require receipts.
    - Any single line item > $25 requires an itemized receipt.
    - Missing required receipts route the claim to MANUAL_REVIEW.
    """

    missing_docs = []
    receipt_issues = []

    for item in claim.get("items", []):
        category = item["category"].lower()
        amount = float(item["amount"])
        receipt_attached = bool(item.get("receipt_attached", False))

        # Determine whether a receipt is mandatory
        receipt_required = (
            category in {"airfare", "lodging"}
            or amount > 25
        )

        if receipt_required and not receipt_attached:
            missing_docs.append(
                f"Receipt required for: {item['description']}"
            )

            receipt_issues.append({
                "description": item["description"],
                "amount": round(amount, 2),
                "reason": "Required receipt missing"
            })

    return {
        "claim_id": claim["claim_id"],
        "receipt_complete": len(missing_docs) == 0,
        "missing_docs": missing_docs,
        "receipt_issues": receipt_issues,
        "policy_refs": [
            "POL-RCT-01",
            "POL-RCT-02"
        ]
    }

In [31]:
# Reimbursement Limit Check

CATEGORY_LIMITS = {
    "meals": {
        "limit": 75.0,
        "unit": "day",
        "policy_ref": "POL-PD-01"
    },
    "lodging": {
        "limit": 200.0,
        "unit": "night",
        "policy_ref": "POL-PD-02"
    },
    "ground_transport": {
        "limit": 50.0,
        "unit": "day",
        "policy_ref": "POL-PD-03"
    }
}


INELIGIBLE_CATEGORIES = {
    "alcohol",
    "minibar",
    "spa",
    "gym",
    "personal_entertainment",
    "personal_shopping",
    "gifts",
    "traffic_fine",
    "penalty",
    "late_fee",
    "personal"
}


def reimbursement_limit_check(claim):
    """
    Calculate reimbursable and deducted amounts.

    Per-day/per-night policies are applied using the quantity
    specified in the claim.
    """

    item_results = []

    total_claimed = 0.0
    total_approved = 0.0
    total_deducted = 0.0

    for item in claim.get("items", []):

        category = item["category"].lower()
        amount = float(item["amount"])

        total_claimed += amount

        # --------------------------------------------------
        # 1. Ineligible expense
        # --------------------------------------------------

        if category in INELIGIBLE_CATEGORIES:

            approved = 0.0
            deducted = amount

            reason = "Ineligible expense category"
            policy_ref = "POL-CAT-02"

        # --------------------------------------------------
        # 2. Business / First Class airfare
        # --------------------------------------------------

        elif (
            category == "airfare"
            and item.get("airfare_class", "").lower()
            in {"business", "first"}
        ):

            # Preserve the amount because the policy exception
            # requires human review.
            approved = amount
            deducted = 0.0

            reason = (
                "Business/first-class airfare requires "
                "Manual Review; amount preserved provisionally"
            )

            policy_ref = "POL-AIR-01"

        # --------------------------------------------------
        # 3. Per-day / per-night reimbursement limits
        # --------------------------------------------------

        elif category in CATEGORY_LIMITS:

            policy = CATEGORY_LIMITS[category]

            limit_per_unit = policy["limit"]

            # Support both "days" and "nights"
            if category == "lodging":
                quantity = float(
                    item.get(
                        "nights",
                        item.get("quantity", 1)
                    )
                )
            else:
                quantity = float(
                    item.get(
                        "days",
                        item.get("quantity", 1)
                    )
                )

            maximum_reimbursable = (
                limit_per_unit * quantity
            )

            approved = min(
                amount,
                maximum_reimbursable
            )

            deducted = max(
                amount - maximum_reimbursable,
                0.0
            )

            if deducted > 0:
                reason = (
                    f"Amount exceeds ${limit_per_unit:.2f} "
                    f"per {policy['unit']} limit"
                )
            else:
                reason = (
                    f"Within ${limit_per_unit:.2f} "
                    f"per {policy['unit']} limit"
                )

            policy_ref = policy["policy_ref"]

        # --------------------------------------------------
        # 4. Other eligible categories
        # --------------------------------------------------

        else:

            approved = amount
            deducted = 0.0

            reason = "Eligible expense"
            policy_ref = "POL-CAT-01"

        total_approved += approved
        total_deducted += deducted

        item_results.append({
            "description": item["description"],
            "category": category,
            "claimed_amount": round(amount, 2),
            "approved_amount": round(approved, 2),
            "deducted_amount": round(deducted, 2),
            "reason": reason,
            "policy_ref": policy_ref
        })

    return {
        "claim_id": claim["claim_id"],
        "total_claimed": round(total_claimed, 2),
        "total_approved": round(total_approved, 2),
        "total_deducted": round(total_deducted, 2),
        "item_results": item_results,
        "policy_refs": sorted({
            result["policy_ref"]
            for result in item_results
        })
    }

In [32]:
# Timeliness Check

from datetime import datetime


def _parse_date(value):
    """Convert YYYY-MM-DD string into a date object."""
    return datetime.strptime(value, "%Y-%m-%d").date()


def timeliness_check(claim):
    """
    Check whether the reimbursement claim was submitted
    within 30 days of the expense date.
    
    For multi-day trips, the trip_start date is used as the
    expense-date baseline unless a specific expense_date exists.
    """

    expense_date = _parse_date(
        claim.get(
            "expense_date",
            claim.get("trip_start")
        )
    )

    submitted_date = _parse_date(
        claim["submitted_date"]
    )

    days_to_submit = (
        submitted_date - expense_date
    ).days

    within_window = (
        0 <= days_to_submit <= 30
    )

    return {
        "claim_id": claim["claim_id"],
        "expense_date": str(expense_date),
        "submitted_date": str(submitted_date),
        "days_to_submit": days_to_submit,
        "within_30_days": within_window,
        "policy_refs": ["POL-TIME-01"],
        "issue": (
            None
            if within_window
            else (
                f"Claim submitted {days_to_submit} "
                f"days after expense date"
            )
        )
    }

In [33]:
for claim in CLAIMS:

    result = timeliness_check(claim)

    print("=" * 60)
    print("Claim:", result["claim_id"])
    print("Expense Date:", result["expense_date"])
    print("Submitted Date:", result["submitted_date"])
    print("Days to Submit:", result["days_to_submit"])
    print("Within 30 Days:", result["within_30_days"])
    print("Issue:", result["issue"])

Claim: CLM-001
Expense Date: 2026-06-10
Submitted Date: 2026-06-20
Days to Submit: 10
Within 30 Days: True
Issue: None
Claim: CLM-002
Expense Date: 2026-06-14
Submitted Date: 2026-06-25
Days to Submit: 11
Within 30 Days: True
Issue: None
Claim: CLM-003
Expense Date: 2026-06-08
Submitted Date: 2026-06-22
Days to Submit: 14
Within 30 Days: True
Issue: None
Claim: CLM-004
Expense Date: 2026-06-16
Submitted Date: 2026-06-28
Days to Submit: 12
Within 30 Days: True
Issue: None
Claim: CLM-005
Expense Date: 2026-06-11
Submitted Date: 2026-06-24
Days to Submit: 13
Within 30 Days: True
Issue: None


In [34]:
# Approval Threshold Check

def approval_threshold_check(reimbursable_amount):
    """
    Determine the approval tier based on the calculated
    reimbursable amount.
    """

    amount = float(reimbursable_amount)

    # Auto approval
    if amount <= 500:

        return {
            "approval_tier": "AUTO_APPROVE",
            "requires_manual_review": False,
            "policy_ref": "POL-APR-01",
            "reason": (
                "Reimbursable amount is <= $500"
            )
        }

    # Manager approval
    elif amount <= 2000:

        return {
            "approval_tier": "MANAGER",
            "requires_manual_review": False,
            "policy_ref": "POL-APR-02",
            "reason": (
                "Reimbursable amount is > $500 "
                "and <= $2,000"
            )
        }

    # Director / Manual Review
    else:

        return {
            "approval_tier": "DIRECTOR",
            "requires_manual_review": True,
            "policy_ref": "POL-APR-03",
            "reason": (
                "Reimbursable amount exceeds $2,000"
            )
        }

In [35]:
for claim in CLAIMS:
    limit_result = reimbursement_limit_check(claim)
    approval_result = approval_threshold_check(
        limit_result["total_approved"]
    )

    print("=" * 60)
    print("Claim:", claim["claim_id"])
    print("Reimbursable Amount:", limit_result["total_approved"])
    print("Approval Tier:", approval_result["approval_tier"])
    print("Manual Review:", approval_result["requires_manual_review"])
    print("Policy:", approval_result["policy_ref"])
    print("Reason:", approval_result["reason"])

Claim: CLM-001
Reimbursable Amount: 1110.0
Approval Tier: MANAGER
Manual Review: False
Policy: POL-APR-02
Reason: Reimbursable amount is > $500 and <= $2,000
Claim: CLM-002
Reimbursable Amount: 0.0
Approval Tier: AUTO_APPROVE
Manual Review: False
Policy: POL-APR-01
Reason: Reimbursable amount is <= $500
Claim: CLM-003
Reimbursable Amount: 840.0
Approval Tier: MANAGER
Manual Review: False
Policy: POL-APR-02
Reason: Reimbursable amount is > $500 and <= $2,000
Claim: CLM-004
Reimbursable Amount: 3000.0
Approval Tier: DIRECTOR
Manual Review: True
Policy: POL-APR-03
Reason: Reimbursable amount exceeds $2,000
Claim: CLM-005
Reimbursable Amount: 75.0
Approval Tier: AUTO_APPROVE
Manual Review: False
Policy: POL-APR-01
Reason: Reimbursable amount is <= $500


In [36]:
# Aggregate Tool Results

def is_airfare_exception(item):
    """
    Detect business/first-class airfare.

    The claim may provide the airfare class explicitly
    through `airfare_class`, or it may be described in
    the item description.
    """
    if item.get("category", "").lower() != "airfare":
        return False

    airfare_class = item.get("airfare_class", "").lower()
    description = item.get("description", "").lower()

    return (
        airfare_class in {"business", "first"}
        or "business-class" in description
        or "first-class" in description
    )


def aggregate_tool_results(claim):
    """
    Execute all deterministic policy tools and combine
    their results into a single evidence package for the LLM.
    """

    # 1. Policy lookup
    policy_result = policy_lookup(claim)

    # 2. Receipt validation
    receipt_result = receipt_completeness_check(claim)

    # 3. Reimbursement calculation
    limit_result = reimbursement_limit_check(claim)

    # 4. Submission timeliness
    timeliness_result = timeliness_check(claim)

    # 5. Approval threshold
    approval_result = approval_threshold_check(
        limit_result["total_approved"]
    )

    # Detect airfare exceptions
    airfare_exceptions = []

    for item in claim.get("items", []):
        if is_airfare_exception(item):
            airfare_exceptions.append(item["description"])

    # Collect reasons that force Manual Review
    manual_review_reasons = []

    if not receipt_result["receipt_complete"]:
        manual_review_reasons.append(
            "Required receipt(s) missing"
        )

    if not timeliness_result["within_30_days"]:
        manual_review_reasons.append(
            "Claim submitted outside the 30-day window"
        )

    if approval_result["requires_manual_review"]:
        manual_review_reasons.append(
            "Reimbursable amount exceeds $2,000"
        )

    if airfare_exceptions:
        manual_review_reasons.append(
            "Business/first-class airfare requires policy exception review"
        )

    return {
        "claim": claim,
        "policy": policy_result,
        "receipt_check": receipt_result,
        "limit_check": limit_result,
        "timeliness_check": timeliness_result,
        "approval_check": approval_result,
        "airfare_exceptions": airfare_exceptions,
        "manual_review_reasons": manual_review_reasons,
        "tool_trace": [
            "policy_lookup",
            "receipt_completeness_check",
            "reimbursement_limit_check",
            "timeliness_check",
            "approval_threshold_check"
        ]
    }

In [37]:
for claim in CLAIMS:
    result = aggregate_tool_results(claim)

    print("=" * 70)
    print("Claim:", claim["claim_id"])
    print("Claimed:", result["limit_check"]["total_claimed"])
    print("Reimbursable:", result["limit_check"]["total_approved"])
    print("Deducted:", result["limit_check"]["total_deducted"])
    print("Receipts Complete:", result["receipt_check"]["receipt_complete"])
    print("Within 30 Days:", result["timeliness_check"]["within_30_days"])
    print("Approval Tier:", result["approval_check"]["approval_tier"])
    print("Airfare Exceptions:", result["airfare_exceptions"])
    print("Manual Review Reasons:", result["manual_review_reasons"])

Claim: CLM-001
Claimed: 1110.0
Reimbursable: 1110.0
Deducted: 0.0
Receipts Complete: True
Within 30 Days: True
Approval Tier: MANAGER
Airfare Exceptions: []
Manual Review Reasons: []
Claim: CLM-002
Claimed: 380.0
Reimbursable: 0.0
Deducted: 380.0
Receipts Complete: True
Within 30 Days: True
Approval Tier: AUTO_APPROVE
Airfare Exceptions: []
Manual Review Reasons: []
Claim: CLM-003
Claimed: 940.0
Reimbursable: 840.0
Deducted: 100.0
Receipts Complete: True
Within 30 Days: True
Approval Tier: MANAGER
Airfare Exceptions: []
Manual Review Reasons: []
Claim: CLM-004
Claimed: 3000.0
Reimbursable: 3000.0
Deducted: 0.0
Receipts Complete: False
Within 30 Days: True
Approval Tier: DIRECTOR
Airfare Exceptions: ['Business-class international airfare']
Manual Review Reasons: ['Required receipt(s) missing', 'Reimbursable amount exceeds $2,000', 'Business/first-class airfare requires policy exception review']
Claim: CLM-005
Claimed: 220.0
Reimbursable: 75.0
Deducted: 145.0
Receipts Complete: False
Wit

In [38]:
# LLM Agent

import json
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)


AGENT_SYSTEM_PROMPT = """
You are a Travel Reimbursement Approval Agent.

Your job is to review a travel reimbursement claim using ONLY:

1. The original claim data.
2. The deterministic tool results.
3. The policy references and evidence contained in those results.

You MUST NOT invent, modify, or assume company policies.

------------------------------------------------------------
DETERMINISTIC TOOL RESULTS ARE THE SOURCE OF TRUTH
------------------------------------------------------------

The following values have already been calculated by
deterministic tools:

- reimbursement amounts
- deductions
- receipt validation
- submission timing
- approval thresholds
- policy references
- policy exceptions

Do NOT recalculate monetary amounts.

------------------------------------------------------------
DECISION RULES
------------------------------------------------------------

APPROVE
Use when:
- All expenses are eligible.
- Required receipts are present.
- Claim is within the 30-day submission window.
- No policy exception exists.
- No manual-review condition exists.

PARTIAL_APPROVE
Use when:
- There are eligible expenses.
- Some eligible expenses exceed policy limits.
- The excess amount is deducted.
- No Manual Review condition exists.

REJECT
Use when:
- All claimed expenses are ineligible.
- Reimbursable amount is $0.
- There is nothing eligible to reimburse.

MANUAL_REVIEW
Use whenever:
- A required receipt is missing.
- Claim is submitted outside the 30-day window.
- Business-class airfare exists.
- First-class airfare exists.
- Reimbursable amount exceeds $2,000.
- Information is ambiguous or conflicting.
- A policy exception may require human approval.

IMPORTANT:

1. Missing receipts must NOT automatically result in REJECT.
2. Business/first-class airfare must NOT automatically be
   deducted. It requires human review because pre-approval
   may exist.
3. Do not change deterministic monetary calculations.
4. The `manual_review_reasons` field is authoritative.
5. Use the policy references provided by the tools.
6. Return JSON only.
"""


def run_llm_agent(tool_results):
    """
    Send the aggregated deterministic tool evidence
    to the LLM and receive a structured recommendation.
    """

    user_prompt = f"""
Review the following travel reimbursement claim.

The deterministic tools have already performed all
policy calculations and validations.

Use their results as the source of truth.

============================================================
CLAIM + TOOL EVIDENCE
============================================================

{json.dumps(tool_results, indent=2, default=str)}

============================================================
OUTPUT REQUIREMENT
============================================================

Return ONLY a valid JSON object using this schema:

{{
    "claim_id": "string",
    "decision": "APPROVE | PARTIAL_APPROVE | REJECT | MANUAL_REVIEW",
    "approved_amount": number,
    "deducted_amount": number,
    "missing_docs": ["string"],
    "policy_refs": ["string"],
    "confidence": number,
    "explanation": "string",
    "tools_used": ["string"]
}}

Rules:

- confidence must be between 0 and 1.
- Use the deterministic approved_amount exactly.
- Use the deterministic deducted_amount exactly.
- Do not recalculate amounts.
- Use the claim_id from the input.
- Use missing documents identified by the receipt tool.
- Use relevant POL-* references.
- If manual_review_reasons is not empty, recommend MANUAL_REVIEW.
- Keep the explanation concise and business-readable.
- Return JSON only.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": AGENT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    content = response.choices[0].message.content

    return json.loads(content)

In [39]:
test_claim = CLAIMS[0]

tool_results = aggregate_tool_results(test_claim)

llm_result = run_llm_agent(tool_results)

print(json.dumps(llm_result, indent=2))

{
  "claim_id": "CLM-001",
  "decision": "APPROVE",
  "approved_amount": 1110.0,
  "deducted_amount": 0.0,
  "missing_docs": [],
  "policy_refs": [
    "POL-CAT-01",
    "POL-PD-01",
    "POL-PD-02",
    "POL-TIME-01",
    "POL-APR-02"
  ],
  "confidence": 1.0,
  "explanation": "All expenses are eligible, receipts are complete, and the claim is within the submission window.",
  "tools_used": [
    "policy_lookup",
    "receipt_completeness_check",
    "reimbursement_limit_check",
    "timeliness_check",
    "approval_threshold_check"
  ]
}


In [40]:
# Decision + Guardrails

ALLOWED_DECISIONS = {
    "APPROVE",
    "PARTIAL_APPROVE",
    "REJECT",
    "MANUAL_REVIEW"
}


def finalize_decision(tool_results, llm_result):
    """
    Apply deterministic policy guardrails over the LLM recommendation
    and produce the final assignment-compliant JSON object.
    """

    limit_result = tool_results["limit_check"]
    receipt_result = tool_results["receipt_check"]
    timeliness_result = tool_results["timeliness_check"]
    approval_result = tool_results["approval_check"]

    # ---------------------------------------------------------
    # Deterministic monetary values
    # ---------------------------------------------------------

    approved_amount = round(
        float(limit_result["total_approved"]),
        2
    )

    deducted_amount = round(
        float(limit_result["total_deducted"]),
        2
    )

    missing_docs = receipt_result["missing_docs"]

    # ---------------------------------------------------------
    # Collect all relevant policy references
    # ---------------------------------------------------------

    policy_refs = set(
        tool_results["policy"]["policy_refs"]
    )

    policy_refs.update(
        receipt_result["policy_refs"]
    )

    policy_refs.update(
        limit_result["policy_refs"]
    )

    policy_refs.update(
        timeliness_result["policy_refs"]
    )

    policy_refs.add(
        approval_result["policy_ref"]
    )

    # ---------------------------------------------------------
    # Hard Manual Review conditions
    # ---------------------------------------------------------

    forced_manual_review = []

    # Missing required receipt
    if missing_docs:
        forced_manual_review.append(
            "Required receipt missing"
        )

    # Late submission
    if not timeliness_result["within_30_days"]:
        forced_manual_review.append(
            "Claim submitted outside 30-day window"
        )

    # Amount exceeds $2,000
    if approval_result["requires_manual_review"]:
        forced_manual_review.append(
            "Reimbursable amount exceeds $2,000 approval threshold"
        )

    # Business / First class airfare
    if tool_results["airfare_exceptions"]:
        forced_manual_review.append(
            "Business/first-class airfare requires policy exception review"
        )

    # ---------------------------------------------------------
    # Final decision hierarchy
    # ---------------------------------------------------------

    if forced_manual_review:

        final_decision = "MANUAL_REVIEW"

    elif approved_amount == 0:

        final_decision = "REJECT"

    elif deducted_amount > 0:

        final_decision = "PARTIAL_APPROVE"

    else:

        final_decision = "APPROVE"

    # ---------------------------------------------------------
    # Build explanation
    # ---------------------------------------------------------

    explanation_parts = []

    if final_decision == "APPROVE":

        explanation_parts.append(
            "All submitted expenses are eligible and compliant "
            "with the applicable policy rules."
        )

    elif final_decision == "PARTIAL_APPROVE":

        explanation_parts.append(
            f"${approved_amount:.2f} is reimbursable after applying "
            "the applicable category limits."
        )

        explanation_parts.append(
            f"${deducted_amount:.2f} is deducted because it exceeds "
            "the applicable reimbursement limits."
        )

    elif final_decision == "REJECT":

        explanation_parts.append(
            "No portion of the submitted claim is reimbursable "
            "under the applicable policy."
        )

    elif final_decision == "MANUAL_REVIEW":

        explanation_parts.append(
            "The claim requires human review because "
            + "; ".join(forced_manual_review)
            + "."
        )

    # Add LLM assessment as supporting reasoning
    llm_explanation = llm_result.get("explanation")

    if llm_explanation:
        explanation_parts.append(
            f"Agent assessment: {llm_explanation}"
        )

    # ---------------------------------------------------------
    # Confidence
    # ---------------------------------------------------------

    try:
        confidence = float(
            llm_result.get("confidence", 0.5)
        )
    except (TypeError, ValueError):
        confidence = 0.5

    confidence = max(
        0.0,
        min(confidence, 1.0)
    )

    # Avoid presenting very high confidence when
    # human review is explicitly required.
    if forced_manual_review:
        confidence = min(
            confidence,
            0.90
        )

    # ---------------------------------------------------------
    # Final assignment-compliant object
    # ---------------------------------------------------------

    final_result = {
        "claim_id": tool_results["claim"]["claim_id"],

        "decision": final_decision,

        "approved_amount": approved_amount,

        "deducted_amount": deducted_amount,

        "missing_docs": missing_docs,

        "policy_refs": sorted(policy_refs),

        "confidence": round(
            confidence,
            2
        ),

        "explanation": " ".join(
            explanation_parts
        ),

        "tools_used": tool_results["tool_trace"]
    }

    return final_result

In [41]:
tool_results = aggregate_tool_results(
    CLAIMS[0]
)

llm_result = run_llm_agent(
    tool_results
)

final_result = finalize_decision(
    tool_results,
    llm_result
)

print(json.dumps(
    final_result,
    indent=2
))

{
  "claim_id": "CLM-001",
  "decision": "APPROVE",
  "approved_amount": 1110.0,
  "deducted_amount": 0.0,
  "missing_docs": [],
  "policy_refs": [
    "POL-AIR-01",
    "POL-APR-01",
    "POL-APR-02",
    "POL-APR-03",
    "POL-CAT-01",
    "POL-CAT-02",
    "POL-PD-01",
    "POL-PD-02",
    "POL-RCT-01",
    "POL-RCT-02",
    "POL-TIME-01"
  ],
  "confidence": 1.0,
  "explanation": "All submitted expenses are eligible and compliant with the applicable policy rules. Agent assessment: All expenses are eligible, receipts are complete, and the claim is within the submission window.",
  "tools_used": [
    "policy_lookup",
    "receipt_completeness_check",
    "reimbursement_limit_check",
    "timeliness_check",
    "approval_threshold_check"
  ]
}


In [42]:
for claim in CLAIMS:

    tool_results = aggregate_tool_results(claim)

    llm_result = run_llm_agent(tool_results)

    final_result = finalize_decision(
        tool_results,
        llm_result
    )

    print("=" * 80)
    print(
        final_result["claim_id"],
        "→",
        final_result["decision"]
    )
    print(
        "Approved:",
        final_result["approved_amount"]
    )
    print(
        "Deducted:",
        final_result["deducted_amount"]
    )
    print(
        "Missing Docs:",
        final_result["missing_docs"]
    )

CLM-001 → APPROVE
Approved: 1110.0
Deducted: 0.0
Missing Docs: []
CLM-002 → REJECT
Approved: 0.0
Deducted: 380.0
Missing Docs: []
CLM-003 → PARTIAL_APPROVE
Approved: 840.0
Deducted: 100.0
Missing Docs: []
CLM-004 → MANUAL_REVIEW
Approved: 3000.0
Deducted: 0.0
Missing Docs: ['Receipt required for: Hotel, 3 nights']
CLM-005 → MANUAL_REVIEW
Approved: 75.0
Deducted: 145.0
Missing Docs: ['Receipt required for: Client dinner for 4 (business development)']
